# Advanced 01 - Dialog, Failover, and 실무 패턴

이 노트북에서는:
- SIP Dialog의 개념과 Kamailio에서의 처리
- Failover 라우팅 패턴
- RTP 프록시(RTPEngine) 연동 개념
- 성능 관련 변수와 플래그

---

## SIP Dialog란?

Dialog는 두 SIP 단말 간의 **지속적인 관계**입니다:

```
Alice          Kamailio          Bob
  |--- INVITE ---->|--- INVITE ---->|
  |<--- 100 ------|                |
  |<--- 180 ------|<--- 180 -------|
  |<--- 200 ------|<--- 200 -------|
  |--- ACK ------>|--- ACK ------->|
  |    [=== 통화 중 (Dialog 활성) ===]   |
  |--- BYE ------>|--- BYE ------->|
  |<--- 200 ------|<--- 200 -------|
```

- Dialog는 Call-ID + From-tag + To-tag로 식별
- `record_route()`를 해야 BYE/re-INVITE가 LB를 경유

## 1. 다이얼로그 내 vs 초기 요청 구분

실무에서 가장 기본이 되는 분기 로직입니다.

In [1]:
# 초기 INVITE 시뮬레이션
%%sip INVITE
From: <sip:1001@example.com>;tag=a1b2c3
To: <sip:1002@example.com>
Contact: <sip:1001@192.168.1.100:5060>

Mock INVITE message created:


INVITE sip:1002@example.com SIP/2.0
From: <sip:1001@example.com>;tag=a1b2c3
To: <sip:1002@example.com>
Call-ID: d174759c@mock
CSeq: 1 INVITE
Contact: <sip:1001@192.168.1.100:5060>


Variables initialized:


  $Ri = "127.0.0.1"


  $Rp = 5060


...

  $ru = "sip:1002@example.com"


  $si = "127.0.0.1"


  $sp = 5060


  $tu = "sip:1002@example.com"


  $ua = ""


In [2]:
# 초기 요청: To tag 없음
if (has_totag()) {
    xlog("In-dialog: relay to existing dialog target");
    # 실제로는 t_relay()만 호출
    t_relay();
} else {
    xlog("Initial request: $(ru{uri.user}) is calling $(tu{uri.user})");
    
    # 라우팅 결정 로직
    if (is_method("INVITE")) {
        record_route();
        lookup("location");
        t_relay();
    }
    if (is_method("REGISTER")) {
        save("location");
    }
}

✗ if (has_totag()) → FALSE → else


[LOG] Initial request: 1002 is calling 1002


✓ if (is_method("INVITE")) → TRUE


→ record_route()


→ lookup("location")


→ t_relay()


✗ if (is_method("REGISTER")) → FALSE


  🔀 if(has_totag()): FALSE → else


## 2. Failover 패턴

한 노드가 응답하지 않을 때 다른 노드로 시도하는 패턴입니다.

```
INVITE → Node1 (응답 없음) → Node2 (성공)
```

Kamailio에서는 `ds_next_dst()`를 사용합니다.

In [3]:
# Failover 시뮬레이션
$var(retry_count) = 0;

# 첫 번째 대상 선택
ds_select_dst("1", "4");
xlog("Try 1: selected destination");

# 실패 시 다음 대상
$var(retry_count) = 1;
ds_next_dst();
xlog("Try 2: next destination (failover)");

# 대상에 실패 표시
ds_mark_dst("p");
xlog("Marked first destination as probing");

$var(retry_count) = 0  (integer)


→ ds_select_dst("1", "4")


[LOG] Try 1: selected destination


$var(retry_count) = 1  (integer)


→ ds_next_dst()


[LOG] Try 2: next destination (failover)


→ ds_mark_dst("p")


[LOG] Marked first destination as probing


Variables changed:


  $var(retry_count): 0 → 1


## 3. RTP 프록시 (RTPEngine) 연동

음성/영상 미디어는 SIP 신호와 **별도 경로**로 전달됩니다:

```
Alice ──[SIP 신호]──> Kamailio ──[SIP 신호]──> Bob
  \                                                  /
   ────────[RTP 미디어]──> RTPEngine <────────────
```

`rtpengine_manage()`는 INVITE/ACK/BYE에 따라 자동으로
offer/answer/delete를 처리합니다.

In [4]:
# RTPEngine 연동 시뮬레이션
%%sip INVITE
From: <sip:1001@192.168.1.100>;tag=rtp1
To: <sip:1002@192.168.1.200>
Contact: <sip:1001@192.168.1.100:5060>

Mock INVITE message created:


INVITE sip:1002@192.168.1.200 SIP/2.0
From: <sip:1001@192.168.1.100>;tag=rtp1
To: <sip:1002@192.168.1.200>
Call-ID: e9d30372@mock
CSeq: 1 INVITE
Contact: <sip:1001@192.168.1.100:5060>


Variables initialized:


  $Ri = "127.0.0.1"


  $Rp = 5060


...

  $ru = "sip:1002@192.168.1.200"


  $si = "127.0.0.1"


  $sp = 5060


  $tu = "sip:1002@192.168.1.200"


  $ua = ""


In [5]:
# 일반적인 RTPEngine 처리 순서
xlog("Step 1: Route to destination");
ds_select_dst("1", "4");

xlog("Step 2: Add Record-Route");
record_route();

xlog("Step 3: Engage RTPEngine");
rtpengine_manage("replace-origin replace-session-connection");

xlog("Step 4: Relay");
t_relay();

[LOG] Step 1: Route to destination


→ ds_select_dst("1", "4")


[LOG] Step 2: Add Record-Route


→ record_route()


[LOG] Step 3: Engage RTPEngine


→ rtpengine_manage("replace-origin replace-session-connection")


[LOG] Step 4: Relay


→ t_relay()


## 4. 플래그와 메모리 관리

Kamailio는 플래그를 통해 요청별 상태를 추적합니다.

| 함수 | 설명 |
|------|------|
| `setflag(n)` | 플래그 n 설정 |
| `resetflag(n)` | 플래그 n 해제 |
| `isflagset(n)` | 플래그 n 확인 |

In [6]:
# 플래그 시뮬레이션
setflag(1);  # 예: "rtp_engine_enabled"

if (isflagset(1)) {
    xlog("RTP engine is enabled for this call");
    rtpengine_manage("replace-origin");
} else {
    xlog("Direct media — no RTP proxy");
}

resetflag(1);

→ setflag(1)


✓ if (isflagset(1)) → TRUE


[LOG] RTP engine is enabled for this call


→ rtpengine_manage("replace-origin")


→ resetflag(1)


  🔀 if(isflagset(1)): TRUE


## 5. 헤더 조작

SIP 메시지의 헤더를 추가/제거하는 것은 실무에서 매우 흔합니다.
예: 추적용 헤더 추가, 불필요한 헤더 제거.

In [7]:
# 헤더 추가
append_hf("X-Call-Source: external");
xlog("Added custom header");

# 헤더 존재 확인
is_present_hf("X-Call-Source");

# 헤더 제거
remove_hf("X-Call-Source");
xlog("Removed custom header");

→ append_hf("X-Call-Source: external")


[LOG] Added custom header


→ is_present_hf("X-Call-Source")


→ remove_hf("X-Call-Source")


[LOG] Removed custom header


## 6. 실행 추적과 디버깅 (v0.2)

`%%trace` 매직으로 코드 실행 경로를 상세히 추적할 수 있습니다.

In [8]:
%%trace
%%sip INVITE
From: <sip:1001@example.com>;tag=trace1
To: <sip:1002@example.com>

In [9]:
%%trace
if (is_method("INVITE")) {
    xlog("INVITE received");
    record_route();
    ds_select_dst("1", "4");
    t_relay();
} else {
    xlog("Not INVITE");
    send_reply(405, "Method Not Allowed");
}

✓ if (is_method("INVITE")) → TRUE [LOG] INVITE received → record_route() → ds_select_dst("1", "4") → t_relay() if (is_method("INVITE") → ✓ TRUE

flowchart TD
    c0["if (is_method('INVITE')"]
    style c0 fill:#e3f2fd,stroke:#2196f3

## 7. 실무 라우팅 템플릿

실제 프로덕션에서 사용하는 `request_route`의 기본 구조:

In [10]:
# 표준 request_route 패턴 시뮬레이션

# 1. 기본 검증
if (!is_method("INVITE|REGISTER|BYE|ACK|CANCEL|OPTIONS")) {
    send_reply(405, "Method Not Allowed");
    exit;
}
xlog("[1] Method validated: $rm");

# 2. 다이얼로그 내 요청 처리
if (has_totag()) {
    xlog("[2] In-dialog request");
    route(RELAY);
}

# 3. REGISTER
if (is_method("REGISTER")) {
    xlog("[3] Registration");
    save("location");
    exit;
}

# 4. INVITE 라우팅
if (is_method("INVITE")) {
    xlog("[4] INVITE routing");
    record_route();
    lookup("location");
    rtpengine_manage("replace-origin");
    t_relay();
}

✗ if (!is_method("INVITE|REGISTER|BYE|ACK|CANCEL|OPTIONS")) → FALSE


[LOG] [1] Method validated: INVITE


✗ if (has_totag()) → FALSE


✗ if (is_method("REGISTER")) → FALSE


✓ if (is_method("INVITE")) → TRUE


...

→ t_relay()


  🔀 if(!is_method("INVITE|REGISTER|BYE|ACK|CANCEL|OPTIONS")): FALSE


  🔀 if(has_totag()): FALSE


  🔀 if(is_method("REGISTER")): FALSE


  🔀 if(is_method("INVITE")): TRUE


In [12]:
# 최종 변수 상태와 변경 이력 확인
%%vars

Variable             Type       Value
------------------------------------------------------------
$Ri                  string     "127.0.0.1"
$Rp                  integer    5060
$ci                  string     "e9d30372@mock"
$cl                  string     "0"
$cs                  string     "1"
$ct                  string     "<sip:1001@192.168.1.100:5060>"
$ft                             "rtp1"
$fu                  string     "sip:1001@192.168.1.100"
$rm                  string     "INVITE"
$ru                  string     "sip:1002@192.168.1.200"
$si                  string     "127.0.0.1"
$sp                  integer    5060
$tu                  string     "sip:1002@192.168.1.200"
$ua                  string     ""
$var(retry_count)    integer    1


In [13]:
%%diff

No variable changes detected.


---

### 실무 패턴 요약

| 패턴 | 핵심 함수 | 설명 |
|------|-----------|------|
| Dialog 분기 | `has_totag()` | 초기 vs 다이얼로그 내 요청 구분 |
| Failover | `ds_select_dst()` + `ds_next_dst()` | 대상 실패 시 다음 대상 |
| RTP 프록시 | `rtpengine_manage()` | 미디어 릴레이 |
| 플래그 | `setflag()` / `isflagset()` | 요청별 상태 추적 |
| 헤더 조작 | `append_hf()` / `remove_hf()` | SIP 헤더 추가/제거 |
| 실행 추적 | `%%trace` | 코드 경로 시각화 |
| 변수 추적 | `%%diff` | 실행 전/후 변수 변화 |

---

### 더 학습하려면

- [Kamailio Wiki](https://www.kamailio.org/wiki/)
- [Kamailio cfg Cookbook](https://www.kamailio.org/wiki/cookbooks/5.5.x/core)
- [RFC 3261 - SIP](https://www.rfc-editor.org/rfc/rfc3261)

이 커리큘럼을 모두 완료하셨습니다! 🎉